# Lesson 2 — Extending the loop

Four mechanisms, one shape: hooks, todos, skills, dynamic system prompts.

Covers `s04` + `s05` + `s07` + `s10`. Narration in `speaker_notes/02_extending_the_loop.md`.

In [ ]:
import json
import subprocess
from pathlib import Path

from llm_client import get_client, DEFAULT_MODEL

client = get_client()

import platform

IS_WINDOWS = platform.system() == "Windows"
SHELL_HINT = (
    "cmd.exe on Windows — use `dir`, `type`, `cd` (no current dir prints "
    "the path), `where`, `findstr`. Avoid Unix-only commands like `ls`, "
    "`pwd`, `cat`, `grep`, `sleep`, `&&`-chained pipelines."
    if IS_WINDOWS else
    "bash on POSIX — `ls`, `pwd`, `cat`, `grep`, etc."
)

def run_bash(command: str) -> str:
    try:
        out = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=30)
        return (out.stdout + out.stderr)[:4000] or "(no output)"
    except Exception as e:
        return f"error: {e}"

BASH_TOOL = {"type": "function", "function": {
    "name": "bash", "description": f"Run a shell command. Shell: {SHELL_HINT}",
    "parameters": {"type": "object",
        "properties": {"command": {"type": "string"}}, "required": ["command"]}}}

## Part 1 — Hooks

Callback registry keyed by event. `PreToolUse` returning a string blocks the tool; `PostToolUse` returning a string transforms the output.

In [ ]:
HOOKS = {
    "PreToolUse": [],   # before running a tool — can block
    "PostToolUse": [],  # after running a tool — can transform output
}

def register_hook(event: str, fn):
    HOOKS[event].append(fn)

def fire_pre(name: str, args: dict):
    for fn in HOOKS["PreToolUse"]:
        result = fn(name, args)
        if result is not None:
            return result   # blocked
    return None

def fire_post(name: str, args: dict, output: str) -> str:
    for fn in HOOKS["PostToolUse"]:
        transformed = fn(name, args, output)
        if transformed is not None:
            output = transformed
    return output

In [ ]:
# Hook #1: re-implement permission denial as a hook
DANGEROUS = ["rm -rf /", "sudo ", "shutdown"]

def permission_hook(name, args):
    if name == "bash":
        cmd = args.get("command", "")
        for pat in DANGEROUS:
            if pat in cmd:
                return f"Permission denied: matched {pat!r}"
    return None

# Hook #2: log every tool call
def logging_hook(name, args):
    print(f"  → tool: {name}({json.dumps(args)[:80]})")
    return None

# Hook #3: truncate huge outputs (PostToolUse)
def truncate_hook(name, args, output):
    if len(output) > 1500:
        return output[:1500] + f"\n…(truncated {len(output)-1500} chars)"
    return None

register_hook("PreToolUse", logging_hook)
register_hook("PreToolUse", permission_hook)
register_hook("PostToolUse", truncate_hook)

Tool dispatch becomes uniform — the permission check lives in a hook now:

In [ ]:
def run_tool_with_hooks(name, args, handler):
    blocked = fire_pre(name, args)
    if blocked is not None:
        return blocked
    output = handler(**args) if handler else f"unknown tool: {name}"
    return fire_post(name, args, str(output))

## Part 2 — TodoWrite

A side-effect-free tool. The model uses it to write down a plan; the harness nags if it goes silent for too many rounds.

In [ ]:
TODOS = []  # session state
rounds_since_todo = 0

TODO_TOOL = {"type": "function", "function": {
    "name": "todo_write",
    "description": "Write or update your plan. Use status: pending|in_progress|completed.",
    "parameters": {"type": "object",
        "properties": {"todos": {"type": "array", "items": {"type": "object",
            "properties": {"content": {"type": "string"},
                           "status": {"type": "string",
                                       "enum": ["pending", "in_progress", "completed"]}},
            "required": ["content", "status"]}}},
        "required": ["todos"]}}}

ICONS = {"pending": " ", "in_progress": "▸", "completed": "✓"}

def run_todo_write(todos):
    global TODOS
    TODOS = todos
    print("  todos:")
    for t in TODOS:
        print(f"    [{ICONS[t['status']]}] {t['content']}")
    return f"updated {len(TODOS)} todos"

## Part 3 — Skills

Index in the system prompt (cheap), body loaded on demand via `load_skill` (expensive). Same two-stage pattern memory will use in lesson 3.

In [ ]:
# Tiny in-memory skill registry — in real Claude Code these are SKILL.md files.
SKILLS = {
    "git-cleanup": {
        "description": "Step-by-step recipe for safely cleaning merged branches.",
        "body": "1. git fetch --prune\n2. git branch --merged main | grep -v main\n3. delete with confirmation",
    },
    "json-format": {
        "description": "How to pretty-print or compact JSON via jq.",
        "body": "Pretty: `jq . file.json`. Compact: `jq -c . file.json`. Filter: `jq '.key' file.json`.",
    },
}

def render_skill_catalog():
    if not SKILLS:
        return ""
    lines = ["You have access to these skills. Call load_skill(name) to read the full body:"]
    for name, s in SKILLS.items():
        lines.append(f"- **{name}**: {s['description']}")
    return "\n".join(lines)

LOAD_SKILL_TOOL = {"type": "function", "function": {
    "name": "load_skill",
    "description": "Load the full body of a skill by name.",
    "parameters": {"type": "object",
        "properties": {"name": {"type": "string"}}, "required": ["name"]}}}

def run_load_skill(name):
    s = SKILLS.get(name)
    return s["body"] if s else f"no such skill: {name}"

## Part 4 — System prompt assembly

Rebuild the system prompt every turn from a `context` dict; cache each section by its content hash so unchanged pieces aren't re-rendered.

In [ ]:
_prompt_cache: dict[str, str] = {}

def assemble_system_prompt(context: dict) -> str:
    sections = [
        "You are an agent running in a Jupyter notebook. Be concise and use tools.",
        f"Workspace: {context.get('cwd', '.')}",
    ]
    if context.get("todos"):
        rendered = "\n".join(f"- [{ICONS[t['status']]}] {t['content']}" for t in context['todos'])
        sections.append("Current todos:\n" + rendered)
    if context.get("skills"):
        sections.append(context["skills"])
    return "\n\n".join(sections)

def get_system_prompt(context: dict) -> str:
    key = json.dumps(context, sort_keys=True, default=str)
    if key not in _prompt_cache:
        _prompt_cache[key] = assemble_system_prompt(context)
    return _prompt_cache[key]

## Putting it together

One loop with all four mechanisms wired in. No new branches in the loop body.

In [ ]:
TOOLS = [BASH_TOOL, TODO_TOOL, LOAD_SKILL_TOOL]
HANDLERS = {
    "bash": run_bash,
    "todo_write": run_todo_write,
    "load_skill": run_load_skill,
}

def agent_loop(user_prompt: str, *, max_turns=12):
    global rounds_since_todo
    messages = [{"role": "user", "content": user_prompt}]
    rounds_since_todo = 0

    for turn in range(max_turns):
        # Per-turn system prompt assembly
        context = {"cwd": str(Path.cwd()), "todos": TODOS, "skills": render_skill_catalog()}
        system = get_system_prompt(context)
        req_messages = [{"role": "system", "content": system}] + messages

        # Nag if model has been silent on todos for 3 rounds
        if rounds_since_todo >= 3:
            req_messages.append({"role": "user", "content":
                "<reminder>You haven't updated your todos in a while. Call todo_write.</reminder>"})
            rounds_since_todo = 0

        resp = client.chat.completions.create(model=DEFAULT_MODEL, messages=req_messages, tools=TOOLS)
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if resp.choices[0].finish_reason != "tool_calls":
            return msg.content

        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            output = run_tool_with_hooks(call.function.name, args, HANDLERS.get(call.function.name))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(output)})

            if call.function.name == "todo_write":
                rounds_since_todo = 0
            else:
                rounds_since_todo += 1

    return "(max_turns reached)"

In [ ]:
answer = agent_loop(
    "You have skills available. Plan a small task with todo_write: "
    "figure out how big this directory is. "
    "Use bash. Mark each todo completed as you go.",
)
print("\nfinal:", answer)

## Recap

Hooks + todos + skills + dynamic prompts. Same loop. Lesson 3 tackles harder context controls: subagents, compaction, memory, recovery.